In [1]:
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv()

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

/workspaces/myGenAI/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
with open("bloodwork.txt", "r") as f:
    blood_report = f.read()
print(blood_report)

Patient: Rajesh Sharma, Age 48, Male
Date: May 7, 2026

COMPLETE BLOOD COUNT (CBC)
--------------------------
Hemoglobin:        15.1 g/dL        (Normal: 13.5–17.5)
Hematocrit:        44%              (Normal: 41–53%)
WBC:               6.8 x10^3/uL     (Normal: 4.5–11.0)
Platelets:         220 x10^3/uL     (Normal: 150–400)

LIPID PANEL
-----------
Total Cholesterol: 238 mg/dL        (Normal: <200)
LDL Cholesterol:   162 mg/dL        (Normal: <100)
HDL Cholesterol:   36 mg/dL         (Normal: >40)
Triglycerides:     188 mg/dL        (Normal: <150)

METABOLIC PANEL
---------------
Glucose (Fasting): 92 mg/dL         (Normal: 70–99)
HbA1c:             5.3%             (Normal: <5.7%)
Creatinine:        1.0 mg/dL        (Normal: 0.7–1.3)
eGFR:              82 mL/min        (Normal: >60)

LIVER FUNCTION
--------------
ALT:               28 U/L           (Normal: 7–40)
AST:               25 U/L           (Normal: 10–40)
Bilirubin Total:   0.8 mg/dL        (Normal: 0.2–1.2)

Reviewing Phys

In [3]:
# Extract blood report and categorize normal and abnormal

extraction_prompt = f"""
You are a senior nurse practitionist and adpet at reading and interpreting the medical reports.
Read the blood report below and extract all test values, classify them as HIGH, LOW or NORMAL.
Format your response as:
- Test Name: value | Status: HIGH/LOW/NORMAL | Reference: range

Blood Report:
{blood_report}
"""

extraction_response = llm.invoke(extraction_prompt)
extracted_values = extraction_response.text
print("=== STAGE 1: EXTRACTED VALUES ===")
print(extracted_values)

=== STAGE 1: EXTRACTED VALUES ===
Here is the interpretation of Rajesh Sharma's blood report:

**COMPLETE BLOOD COUNT (CBC)**
- Hemoglobin: 15.1 g/dL | Status: NORMAL | Reference: 13.5–17.5
- Hematocrit: 44% | Status: NORMAL | Reference: 41–53%
- WBC: 6.8 x10^3/uL | Status: NORMAL | Reference: 4.5–11.0
- Platelets: 220 x10^3/uL | Status: NORMAL | Reference: 150–400

**LIPID PANEL**
- Total Cholesterol: 238 mg/dL | Status: HIGH | Reference: <200
- LDL Cholesterol: 162 mg/dL | Status: HIGH | Reference: <100
- HDL Cholesterol: 36 mg/dL | Status: LOW | Reference: >40
- Triglycerides: 188 mg/dL | Status: HIGH | Reference: <150

**METABOLIC PANEL**
- Glucose (Fasting): 92 mg/dL | Status: NORMAL | Reference: 70–99
- HbA1c: 5.3% | Status: NORMAL | Reference: <5.7%
- Creatinine: 1.0 mg/dL | Status: NORMAL | Reference: 0.7–1.3
- eGFR: 82 mL/min | Status: NORMAL | Reference: >60

**LIVER FUNCTION**
- ALT: 28 U/L | Status: NORMAL | Reference: 7–40
- AST: 25 U/L | Status: NORMAL | Reference: 10–40


In [4]:
#Give me diet plan
diet_prompt = f"""
You are a clinical nutritionist.

Based on the blood work analysis below, write:
1. A short health summary in 4-5 lines explaining the patient's condition in simple language
2. A short, practical diet plan having only two sections (1) Foods to avoid (2) Foods to eat more of. 
   Do not include any other sections in diet plan.

Blood Work Analysis:
{extracted_values}
"""

diet_response = llm.invoke(diet_prompt)

print("=== STAGE 2: HEALTH SUMMARY & DIET PLAN ===")
print(diet_response.text)

=== STAGE 2: HEALTH SUMMARY & DIET PLAN ===
Here is an analysis of your blood work and a tailored dietary plan:

**1. Health Summary**

Rajesh, your overall health markers, including blood sugar, kidney, and liver functions, are currently stable. However, your lipid profile shows some concerns: your "bad" cholesterol (LDL) and triglycerides are elevated, while your "good" cholesterol (HDL) is lower than ideal. This pattern suggests an increased risk for heart-related issues, making targeted dietary and lifestyle changes very important for your long-term health.

**2. Diet Plan**

**Foods to Avoid:**
*   Fried foods, processed snacks, and fast food items.
*   Fatty cuts of red meat, full-fat dairy products (like full-cream milk, butter, ghee in excess).
*   Baked goods, pastries, sweets, and sugary drinks.
*   Foods high in trans fats (often listed as "partially hydrogenated oils" on labels).
*   Excessive refined carbohydrates like white bread and white rice.

**Foods to Eat More Of:**